# 1. Initializations

## 1.1 General CPU/GPU Checks (NVIDIA cards)

In [ ]:
import os
import time
print(f'Path [{os.environ["PATH"]}]')

# torch_test.py
import torch
print(f"✅ Torch CUDA available: {torch.cuda.is_available()}")
print(f"🖥️ Device: {torch.cuda.get_device_name(0)}")

# tf_test.py
import tensorflow as tf
print("✅ TF GPU:", tf.config.list_physical_devices("GPU"))

# Test Tensor flow
with tf.device('/GPU:0'):
    print("Lancement benchmark sur GPU...")
    start = time.time()
    a = tf.random.normal([10000, 10000])
    b = tf.random.normal([10000, 10000])
    c = tf.matmul(a, b)
    tf.print("Fin multiplication")
    tf.print("Durée:", time.time() - start, "secondes")

# Comparatif CPU
with tf.device('/CPU:0'):
    print("Lancement benchmark sur CPU...")
    start = time.time()
    a = tf.random.normal([10000, 10000])
    b = tf.random.normal([10000, 10000])
    c = tf.matmul(a, b)
    tf.print("Fin multiplication")
    tf.print("Durée:", time.time() - start, "secondes")

## 1.2 General imports

In [ ]:
# Pour la manipulation de tableaux et Dataframes
import numpy as np
import pandas as pd

# Pour les modèles et leur preprocessing
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# Pour la visualisation des performances
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

# Pour instancier une couche Dense et modèle séquentiel
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense

In [ ]:
import smartcheck.dataframe_common as dfc

# 2. Loading and Data Enrichment

In [ ]:
df_iris_raw = dfc.load_dataset_from_config('iris_data', sep=',')

if df_iris_raw is not None and isinstance(df_iris_raw, pd.DataFrame):
    df_iris = df_iris_raw.copy()

In [ ]:
df_iris.info()

In [ ]:
X = df_iris.drop(columns='species')
y = df_iris['species']

In [ ]:
pre_le = LabelEncoder()
y_encoded = pre_le.fit_transform(y)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=1/3, random_state=42) 

In [ ]:
display(X_train)

# 3. Deep learning

#### Creation & Compilation

In [ ]:
# Création du modèle séquentiel
nn_tfkeras_seq = Sequential()

# Ajout des couches dans l'ordre
nn_tfkeras_seq.add(Dense(units=10, activation="tanh", input_shape=(4,)))
nn_tfkeras_seq.add(Dense(units=8, activation="tanh"))
nn_tfkeras_seq.add(Dense(units=6, activation="tanh"))
nn_tfkeras_seq.add(Dense(units=3, activation="softmax"))

In [ ]:
nn_tfkeras_seq.summary()

In [ ]:
nn_tfkeras_seq.compile(
    loss="sparse_categorical_crossentropy",              
    optimizer="adam",
    metrics=["accuracy"]
)

#### Entrainement et Prédiction

In [ ]:
history = nn_tfkeras_seq.fit(X_train, y_train, epochs=500, batch_size=32, validation_split=0.1)

In [ ]:
y_test_prob = nn_tfkeras_seq.predict(X_test)

In [ ]:
# les predictions peuvent servir en utilisant l'argmax par colonne (indice de la colonne ayant la plus forte proba)
# à déterminer la classe prédite (si colonne 0 = classe 0, colonne 1 = classe 1, etc...)
# vu qu'on a encodé y avec des label de 0 à X
y_test_pred = np.argmax(y_test_prob, axis=1)

#### Visualisation des résultats

In [ ]:
print(classification_report(y_test, y_test_pred))
sns.heatmap(confusion_matrix(y_test, y_test_pred), cmap='Blues', cbar=False, annot=True)

In [ ]:
plt.figure()
plt.plot(history.history['loss'], label='Loss (entraînement)')
plt.plot(history.history['val_loss'], label='Loss (validation)')
plt.title('Courbe de la perte par époque')
plt.xlabel('Épochs')
plt.ylabel('Perte')
plt.legend()
plt.show()